# Pantry Vision: fine-tune the detector (Colab, free GPU)

Input: the owner's labelled photos exported from Label Studio in **YOLO** format to a private Drive folder
(`images/`, `labels/`, `classes.txt`; class names are item keys). Output: `pantry-finetuned.onnx` plus its JSON
sidecar, which the service loads exactly like the zero-shot model. Photos stay in the private Drive folder
(constitution v1.1.0, Principle IV). Runtime: Colab, GPU (T4).

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/pantry-vision/labelled'  # Label Studio YOLO export
OUT = '/content/drive/MyDrive/pantry-vision/models'

In [ ]:
%pip install -q ultralytics onnx onnxslim
from ultralytics import settings

settings.update({'sync': False})  # no usage analytics

In [ ]:
import random
from pathlib import Path

data = Path(DATA)
classes = (data / 'classes.txt').read_text().split()
images = sorted(p for p in (data / 'images').iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
random.Random(0).shuffle(images)
cut = int(len(images) * 0.8)
Path('train.txt').write_text('\n'.join(map(str, images[:cut])))
Path('val.txt').write_text('\n'.join(map(str, images[cut:])))
names = '\n'.join(f'  {i}: {c}' for i, c in enumerate(classes))
Path('data.yaml').write_text(f'train: {Path("train.txt").resolve()}\nval: {Path("val.txt").resolve()}\nnames:\n{names}\n')
print(len(images), 'photos,', len(classes), 'classes, train', cut, 'val', len(images) - cut)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
result = model.train(data='data.yaml', epochs=100, imgsz=640, batch=16, patience=20, seed=0)
best = Path(result.save_dir) / 'weights' / 'best.pt'
print(best)

In [ ]:
import hashlib
import json
import shutil

onnx = Path(YOLO(str(best)).export(format='onnx', imgsz=640, simplify=True))
Path(OUT).mkdir(parents=True, exist_ok=True)
target = Path(OUT) / 'pantry-finetuned.onnx'
shutil.copy(onnx, target)
sidecar = {
    'model': target.name,
    'sha256': hashlib.sha256(target.read_bytes()).hexdigest(),
    'source_weights': 'yolo11s.pt fine-tuned',
    'imgsz': 640,
    'classes': classes,
}
target.with_suffix('.json').write_text(json.dumps(sidecar, indent=2))
print('saved', target, 'and its .json')

Next, on the owner's machine: copy both files into `models/`, then
`python -m uv run python eval/run_eval.py --data <held-out set> --model models/pantry-finetuned.onnx`.
Ship only with before and after numbers in the README (Principle III).